# PZT-4 Piezoelectric PINN Solver
This notebook converts the original Python script into a structured Jupyter Notebook workflow for solving the coupled thermo-piezoelectric problem using Physics-Informed Neural Networks (PINNs).

## 1. Import Required Libraries

In [1]:
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import torch

import src.config as cfg
from src.temperature import solve_temperature
from src.thermal_loading import (
    compute_AB,
    compute_tau0_field,
    make_tau0_interpolator,
)

from src.network import MechanicsNet
from src.trainer import Trainer

from src.postprocess import (
    plot_tau0_field,
    plot_temperature,
    plot_sif,
    plot_loss_history,
    plot_displacement_field,
    compute_sif,
    plot_tau11_near_tip,
)


## 2. Define Simulation Parameters

In [2]:
# Simulation parameters

gamma = cfg.GAMMA

adam_iter = cfg.MAX_ITER_ADAM
lbfgs_iter = cfg.MAX_ITER_LBFGS

nx_temp = 120
nt_temp = 300

figures_dir = "figures"

n_int = cfg.N_INTERIOR
n_bc = cfg.N_BOUNDARY

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64

print(f"Running on {device} | gamma = {gamma}")


Running on cpu | gamma = 0.8


## 3. Solve Fractional Heat Equation

In [3]:
x3_grid, t_grid, T_field = solve_temperature(
    N_x=nx_temp,
    N_t=nt_temp,
    gamma=gamma,
)

plot_temperature(
    x3_grid,
    t_grid,
    T_field,
    save_dir=figures_dir,
)


Saved temperature_field.png


## 4. Compute Thermal Loading Field

In [4]:
# Compute balancing coefficients
A, B = compute_AB(x3_grid, T_field)

# Compute thermal loading field
tau0_field = compute_tau0_field(
    x3_grid,
    t_grid,
    T_field,
    A,
    B,
)

# Plot thermal loading field
plot_tau0_field(
    x3_grid,
    t_grid,
    tau0_field,
)


Saved tau0_field.png


## 5. Create Thermal Interpolator

In [5]:
tau0_fn = make_tau0_interpolator(
    x3_grid,
    t_grid,
    tau0_field,
)


## 6. Construct PINN Model

In [6]:
net = MechanicsNet(cfg.MECH_LAYERS)

trainer = Trainer(
    net,
    tau0_fn,
    device=device,
    dtype=dtype,
    n_interior=n_int,
    n_boundary=n_bc,
    max_iter_adam=adam_iter,
    max_iter_lbfgs=lbfgs_iter,
)


## 7. Train the PINN

In [7]:
trainer.train()

# Save model checkpoint
checkpoint_path = "checkpoints/model.pt"
trainer.save(checkpoint_path)

print("Training completed.")


Device: cpu  |  dtype: torch.float64
Parameters: 8,003
Phase 1 — Adam


c:\Users\tdiks\Documents\PhD NITH\research paper\6th paper\thermo\PINN\src\trainer.py:111: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:839.)
  if not math.isfinite(grad_norm) or not math.isfinite(float(loss)):



Epoch : 10/500
Total Loss : 3.670837e-02

--- PDE ---
PDE Loss   : 8.463849e-14

--- Boundary Losses ---
BC1 : 3.557389e-03
BC2 : 9.844383e-12
BC3 : 1.211216e-05
BC4 : 4.690290e-17
BC5 : 1.013360e-04

BC Total   : 3.670837e-03

--- Other ---
Grad Norm  : 9.809380e-02
Param Norm : 1.263457e+01
Elapsed    : 6.9 s


Epoch : 20/500
Total Loss : 3.264831e-02

--- PDE ---
PDE Loss   : 3.445560e-13

--- Boundary Losses ---
BC1 : 2.959202e-03
BC2 : 8.991464e-12
BC3 : 1.143361e-05
BC4 : 4.489164e-17
BC5 : 2.941951e-04

BC Total   : 3.264831e-03

--- Other ---
Grad Norm  : 6.438081e-02
Param Norm : 1.263566e+01
Elapsed    : 14.4 s


Epoch : 30/500
Total Loss : 2.913562e-02

--- PDE ---
PDE Loss   : 7.746567e-13

--- Boundary Losses ---
BC1 : 2.379116e-03
BC2 : 8.584964e-12
BC3 : 1.013129e-05
BC4 : 4.071374e-17
BC5 : 5.243156e-04

BC Total   : 2.913562e-03

--- Other ---
Grad Norm  : 4.123023e-02
Param Norm : 1.263779e+01
Elapsed    : 23.3 s


Epoch : 40/500
Total Loss : 2.906171e-02

--- PDE --

## 8. Plot Loss History

In [8]:
plot_loss_history(
    trainer.history,
    save_dir=figures_dir,
)


Saved loss_history.png


## 9. Compute Stress Intensity Factors

In [9]:
t_plot = np.linspace(0.1, cfg.T_MAX, 500)
t_plot = t_plot[t_plot <= cfg.T_MAX]

K_Ia, K_Ib = compute_sif(
    net,
    t_plot,
    device,
    dtype,
)

plot_sif(
    t_plot,
    K_Ia,
    K_Ib,
    save_dir=figures_dir,
)

print(f"K_Ia at final time = {K_Ia[-1]:.4f} MPa√m")
print(f"K_Ib at final time = {K_Ib[-1]:.4f} MPa√m")


Saved K_Ia_vs_time.png
Saved K_Ib_vs_time.png
K_Ia at final time = -54062.8658 MPa√m
K_Ib at final time = -53921.3956 MPa√m


## 10. Plot Displacement Fields

In [10]:
for tv in t_plot[:3]:
    plot_displacement_field(
        net,
        tv,
        device,
        dtype,
        save_dir=figures_dir,
    )

print("Displacement field plots generated.")


Saved fields_t0p10ppng
Saved fields_t0p18ppng
Saved fields_t0p26ppng
Displacement field plots generated.


# Stresses near crack tip

In [11]:
plot_tau11_near_tip(
    net,
    t_val=t_plot[len(t_plot)//2],
    device=device,
    dtype=dtype,
    tip="a",
)

plot_tau11_near_tip(
    net,
    t_val=t_plot[len(t_plot)//2],
    device=device,
    dtype=dtype,
    tip="b",
)

Saved tau11_tip_a.png
Saved tau11_tip_b.png


## 11. Final Remarks
This notebook provides a complete workflow for:
- Solving the fractional heat equation
- Computing thermoelastic loading
- Training the PINN model
- Computing stress intensity factors
- Generating post-processing figures